In [1]:
%matplotlib inline



# 神经网络

神经网络可以使用 `torch.nn` 包来构建。

现在你已经了解了 `autograd` 的基本概念，`nn` 依赖 `autograd` 来定义模型并计算其梯度。一个 `nn.Module` 包含各个层（layers）以及一个 `forward(input)` 方法，该方法返回模型的输出（`output`）。

例如，下面是一个用于分类数字图像的网络：

![convnet](/_static/img/mnist.png)

这是一个简单的**前馈网络**（feed-forward network）。它接收输入，将输入依次通过多个层进行处理，最终生成输出。

神经网络的典型训练过程如下：

1. 定义一个具有可学习参数（或权重）的神经网络。
2. 遍历输入数据集。
3. 通过网络处理输入。
4. 计算损失（输出的预测值与真实值之间的差距）。
5. 将梯度反向传播到网络的参数中。
6. 更新网络的权重，通常使用简单的更新规则：
   $$
   \text{weight} = \text{weight} - \eta \cdot \text{gradient}
   $$
   其中：
   - $\text{weight}$：权重参数。
   - $\eta$：学习率（learning rate）。
   - $\text{gradient}$：损失对权重的梯度。

## 定义网络

让我们来定义这个网络：


1. 什么是卷积（Convolution）？

卷积就像是用一个小放大镜在图片上滑来滑去，仔细观察图片的每一小块，提取出重要的“特征”，比如边缘、角落、纹理等。这些特征帮助计算机“看懂”图片的内容。

- 卷积在做什么？
- 卷积操作用一个小的“过滤器”（也叫卷积核，比如 3x3 的小矩阵）扫描图片。
- 每次扫描时，过滤器会和图片的一小块区域做数学运算（点积），生成一个数字，表示这个区域的特征强度。
- 扫描完整个图片后，得到一张新的图，叫特征图（Feature Map），它突出显示了图片中某些特征（比如边缘）。
```
设你有一张 5x5 的灰度图片
1 0 0 0 1
0 1 0 1 0
0 0 1 0 0
0 1 0 1 0
1 0 0 0 1
用一个 3x3 的卷积核（过滤器）
1  1  1
0  0  0
-1 -1 -1
最终，特征图
0  1  0
-1 2  1
0  -1 0

28x28 用 3x3 卷积核扫描，生成特征图尺寸约 26x26(因为 28 - 3 + 1 = 26，卷积核“吃掉”边缘）。)
```

2. 什么是池化（Pooling）？
- 池化操作把特征图分成小区域（比如 2x2），对每个区域做一次“总结”，输出一个数字。
- 最常见的是最大池化（Max Pooling）：从每个小区域挑最大的值。
- 结果是特征图尺寸变小（比如 26x26 → 13x13），但保留了最重要的特征。
```
有一张 4x4 的特征图
4  2  1  0
3  5  2  1
1  2  6  3
0  1  3  4
用 2x2 的最大池化（步幅 2，意思是每块区域不重叠）：
得到新的特征图（2x2）
5  2
2  6

对每张 26x26 的特征图做 2x2 最大池化，尺寸缩小到 13x13（26 ÷ 2）。
```


In [ ]:
import torch
import torch.nn as nn #卷积层、全连接层
import torch.nn.functional as F #激活函数、池化（缩小图片）


class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__() #继承后，Net 可以利用 PyTorch 的自动求导、参数管理和模型保存等功能。
        # kernel
        # 卷积层就像一个“扫描仪”，它用一个小窗口（叫卷积核）在图片上滑来滑去，提取图片的特征
        self.conv1 = nn.Conv2d(1, 6, 3) #输入通道数。通道数表示图片的。“颜色深度”1，说明输入是灰度图片
        # 3：卷积核大小，3x3 的小方块。每次扫描时，窗口看 3x3 的像素区域。
# 1 input image channel, 6 output channels, 3x3 square convolution
        self.conv2 = nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        # 积层处理完图片后，特征图被“展平”成一个长向量
        self.fc1 = nn.Linear(16 * 6 * 6, 120)  # 6*6 from image dimension 
        # 压缩成 120 个数字，减少信息量但保留关键特征
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

# 为什么是 16 * 6 * 6？

# 卷积和池化（后面会讲池化）会把图片尺寸变小。假设输入图片是 28x28：
# 第一次卷积后，尺寸变成 26x26（因为 3x3 卷积核“吃掉”了边缘）。
# 第一次池化（缩小一半）后，变成 13x13。
# 第二次卷积后，变成 11x11。
# 第二次池化后，变成 6x6（四舍五入）。
# 最后有 16 张 6x6 的特征图，展平后就是 16 * 6 * 6 = 576。

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2)) #F.relu(...)：ReLU 激活函数，把负数变成 0，正数不变
# F.max_pool2d(..., (2, 2))：最大池化，像一个“缩图机”，用 2x2 的窗口扫描特征图，选出窗口里最大的值，尺寸缩小一半（26x26 → 13x13）
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2) #池化的 (2, 2) 简写为 2，意思相同
        x = x.view(-1, self.num_flat_features(x)) #self.num_flat_features(x)：计算特征图的总数量
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x #返回形状为 (batch_size, 10) 的张量，每行是 10 个类别的得分。

    def num_flat_features(self, x):
        size = x.size()[1:]  #x.size()[1:]：获取张量的形状（除了批量维度）。例如，x 的形状是 (batch_size, 16, 6, 6)，size 就是 (16, 6, 6)。
        num_features = 1
        for s in size:
            num_features *= s
        return num_features


net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=576, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)



神经网络定义与参数

你只需定义 `forward` 函数，而 `backward` 函数（用于计算梯度）会通过 `autograd` 自动为你定义。在 `forward` 函数中，你可以使用任何张量操作。

模型的可学习参数可以通过 `net.parameters()` 获取。



In [ ]:
params = list(net.parameters())
print(len(params))
print(params[0].size())  # conv1's .weight 第一个参数是 conv1 的权重矩阵

10
torch.Size([6, 1, 3, 3])



尝试随机的 32x32 输入

注意：此网络（LeNet）的预期输入尺寸为 32x32。要在 MNIST 数据集上使用此网络，请将数据集中的图像调整为 32x32 大小。

In [4]:
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

tensor([[-0.0125,  0.1669, -0.0183, -0.0262, -0.0363, -0.0834,  0.0346, -0.0302,
         -0.0276, -0.0211]], grad_fn=<AddmmBackward>)




将所有参数的梯度缓冲区清零，并使用随机梯度进行反向传播：

In [ ]:
net.zero_grad() #清空网络中所有参数的梯度。
out.backward(torch.randn(1, 10)) #out 不是标量，而是一个形状为 (1, 10) 的张量。直接调用 out.backward() 会报错



 注意事项与 PyTorch 类回顾

<div class="alert alert-info"><h4>注意</h4><p>`torch.nn` 仅支持小批量（mini-batch）输入。整个 `torch.nn` 包只支持输入为样本的小批量，而不是单个样本。

例如，`nn.Conv2d` 需要一个 4D 张量，形状为 `nSamples x nChannels x Height x Width`。

如果只有一个样本，可以使用 `input.unsqueeze(0)` 添加一个伪批次维度。</p></div>

在继续之前，让我们回顾一下你目前见过的所有类。

 回顾
- **`torch.Tensor`**：一个支持自动求导操作（如 `backward()`）的*多维数组*，同时*保存张量的梯度*。
- **`nn.Module`**：神经网络模块，*方便封装参数*，提供将参数移动到 GPU、导出、加载等辅助功能。
- **`nn.Parameter`**：一种特殊的张量，当作为 `Module` 的属性时，*自动注册为参数*。
- **`autograd.Function`**：实现自动求导操作的*前向和反向定义*。每个张量操作至少创建一个 `Function` 节点，连接到生成该张量的函数，并*编码其历史*。

 当前已覆盖内容
- 定义神经网络
- 处理输入并调用反向传播

 仍需覆盖内容
- 计算损失
- 更新网络权重

损失函数

损失函数接收一对输入（输出和目标值），并计算一个值，估计输出与目标值之间的差距。

`nn` 包中提供了多种不同的[损失函数](https://pytorch.org/docs/nn.html#loss-functions)。一个简单的损失函数是 `nn.MSELoss`，它计算输入和目标值之间的均方误差。

例如：


In [6]:
output = net(input)
target = torch.randn(10)  # a dummy target, for example
target = target.view(1, -1)  # make it the same shape as output
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(0.9962, grad_fn=<MseLossBackward>)



 反向传播与计算图

现在，如果你沿着 `loss` 的反向路径，通过其 `.grad_fn` 属性进行追溯，你将看到一个如下所示的计算图：

::

    input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
          -> view -> linear -> relu -> linear -> relu -> linear
          -> MSELoss
          -> loss

因此，当我们调用 `loss.backward()` 时，整个计算图会相对于损失进行求导，图中所有设置了 `requires_grad=True` 的张量都会将其 `.grad` 张量累积对应的梯度。

为了说明这一点，让我们追溯几个反向步骤：


In [7]:
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU



反向传播

要反向传播误差，我们只需调用 `loss.backward()`。但需要先清零现有的梯度，否则新计算的梯度会累加到已有的梯度上。

现在我们将调用 `loss.backward()`，并查看第一层卷积层（`conv1`）的偏置梯度在反向传播前后的变化。

In [8]:
net.zero_grad()     # zeroes the gradient buffers of all parameters

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
tensor([0., 0., 0., 0., 0., 0.])
conv1.bias.grad after backward
tensor([ 0.0022, -0.0084, -0.0044,  0.0091, -0.0113,  0.0077])




 神经网络损失函数与权重更新

现在，我们已经了解了如何使用损失函数。

 后续阅读

神经网络包（`torch.nn`）包含多种模块和损失函数，这些是构建深度神经网络的基础组件。完整的列表和文档可以在[这里](https://pytorch.org/docs/nn)找到。

 剩余内容

唯一尚未学习的内容是：

- 更新网络的权重

更新权重

实践中最简单的更新规则是**随机梯度下降**（Stochastic Gradient Descent, SGD）：

$$
\text{weight} = \text{weight} - \eta \cdot \text{gradient}
$$

我们可以使用简单的 Python 代码实现这一规则：

.. code:: python

    learning_rate = 0.01
    for f in net.parameters():
        f.data.sub_(f.grad.data * learning_rate)

然而，在使用神经网络时，你可能希望使用不同的更新规则，例如 SGD、Nesterov-SGD、Adam、RMSProp 等。为了支持这些方法，我们构建了一个小型包：torch.optim，它实现了所有这些优化方法。使用它非常简单：



In [9]:
import torch.optim as optim

# create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()    # Does the update



> **注意**：
>
> 请注意，必须使用 `optimizer.zero_grad()` 手动将梯度缓冲区清零。这是因为梯度会累积，如[反向传播](#反向传播)部分所述。